## Assignment 2: Improve and Evaluate RAG System

In [1]:
from pathlib import Path
import sys


current_folder = Path.cwd()


if current_folder.name == "notebooks":
    PROJECT_ROOT = current_folder.parent
else:
    PROJECT_ROOT = current_folder


if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


print("Project Root:", PROJECT_ROOT)

Project Root: C:\Users\Nafisa\PycharmProjects\rag-assignment-2


## Task 1 — Load the Documents

In [2]:
from src.ingestion import (
    load_documents,
    show_document_samples
)


documents = load_documents()


show_document_samples(documents)

Total documents loaded: 10

Document Samples

File Name: artificial_intelligence.txt
Preview: Artificial Intelligence, commonly called AI, is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks may inclu
----------------------------------------------------------------------
File Name: data_science.txt
Preview: Data science is the process of collecting, cleaning, analyzing, and interpreting data to solve problems and support decisions. It combines statistics, programming, mathematics, and knowledge about a p
----------------------------------------------------------------------
File Name: football.txt
Preview: Football is one of the most popular sports in the world. It is played between two teams, and each team normally has eleven players. The main objective is to score goals by moving the ball into the opp
----------------------------------------------------------------------
File Name: health.tx

## Task 2 — Compare Different Chunk Sizes

This experiment compares three chunk configurations:


| Experiment | Chunk Size | Overlap |
|---|---:|---:|
| A | 300 | 50 |
| B | 500 | 50 |
| C | 800 | 100 |

In [3]:
from src.chunking import (
    chunk_all_documents,
    show_chunk_samples
)


chunks_300 = chunk_all_documents(
    documents=documents,
    chunk_size=300,
    chunk_overlap=50
)

print("\nExperiment A Sample")
show_chunk_samples(
    chunks=chunks_300,
    number_of_samples=1
)


chunks_500 = chunk_all_documents(
    documents=documents,
    chunk_size=500,
    chunk_overlap=50
)

print("\nExperiment B Sample")
show_chunk_samples(
    chunks=chunks_500,
    number_of_samples=1
)


chunks_800 = chunk_all_documents(
    documents=documents,
    chunk_size=800,
    chunk_overlap=100
)

print("\nExperiment C Sample")
show_chunk_samples(
    chunks=chunks_800,
    number_of_samples=1
)

Chunk Size: 300
Chunk Overlap: 50
Total Chunks Created: 161

Experiment A Sample

Chunk Samples

File Name: artificial_intelligence.txt
Chunk Number: 0
Text:
Artificial Intelligence, commonly called AI, is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks may include understanding language, recognizing images, solving problems, making decisions, learning from
----------------------------------------------------------------------
Chunk Size: 500
Chunk Overlap: 50
Total Chunks Created: 103

Experiment B Sample

Chunk Samples

File Name: artificial_intelligence.txt
Chunk Number: 0
Text:
Artificial Intelligence, commonly called AI, is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks may include understanding language, recognizing images, solving problems, making decisions, learning from experience, and gen

### Create Qdrant Collections for the Chunk Experiments

In [4]:
from src.rag_pipeline import prepare_rag_database


chunk_experiments = [
    {
        "experiment": "A",
        "collection_name": "rag_documents_chunk_300",
        "chunk_size": 300,
        "chunk_overlap": 50
    },
    {
        "experiment": "B",
        "collection_name": "rag_documents_chunk_500",
        "chunk_size": 500,
        "chunk_overlap": 50
    },
    {
        "experiment": "C",
        "collection_name": "rag_documents_chunk_800",
        "chunk_size": 800,
        "chunk_overlap": 100
    }
]


# Build or reuse each Qdrant collection
for experiment in chunk_experiments:

    print("\n" + "=" * 70)

    print(
        "Experiment:",
        experiment["experiment"]
    )

    print(
        "Collection:",
        experiment["collection_name"]
    )

    experiment_chunks = prepare_rag_database(
        collection_name=experiment["collection_name"],
        chunk_size=experiment["chunk_size"],
        chunk_overlap=experiment["chunk_overlap"],
        rebuild=False
    )

    print(
        "Total Chunks:",
        len(experiment_chunks)
    )


Experiment: A
Collection: rag_documents_chunk_300
Total documents loaded: 10
Chunk Size: 300
Chunk Overlap: 50
Total Chunks Created: 161
Expected number of points: 161
Creating embeddings for all chunks...
Total embeddings created: 161
Embedding vector size: 1536
Detected vector size: 1536
Collection created: rag_documents_chunk_300
Total points stored in Qdrant: 161
RAG database preparation completed.
Total Chunks: 161

Experiment: B
Collection: rag_documents_chunk_500
Total documents loaded: 10
Chunk Size: 500
Chunk Overlap: 50
Total Chunks Created: 103
Expected number of points: 103
Existing points in Qdrant: 103
Collection is already complete.
Embedding API calls were skipped.
Total Chunks: 103

Experiment: C
Collection: rag_documents_chunk_800
Total documents loaded: 10
Chunk Size: 800
Chunk Overlap: 100
Total Chunks Created: 67
Expected number of points: 67
Creating embeddings for all chunks...
Total embeddings created: 67
Embedding vector size: 1536
Detected vector size: 1536
C

### Test the Same Question with Different Chunk Sizes

The same question is used for all three chunk configurations.  
For each experiment, the system retrieves the Top-3 chunks, prints their similarity scores, and generates a final answer using the retrieved context.

In [6]:
from src.retrieval import (
    retrieve_chunks,
    show_retrieved_chunks
)

from src.prompt import create_prompt
from src.generation import generate_answer


chunk_test_question = (
    "What is supervised learning and how should "
    "a model be evaluated on unseen data?"
)


chunk_experiment_results = []


for experiment in chunk_experiments:

    print("\n" + "=" * 80)

    print(
        "Experiment:",
        experiment["experiment"]
    )

    print(
        "Chunk Size:",
        experiment["chunk_size"]
    )

    print(
        "Chunk Overlap:",
        experiment["chunk_overlap"]
    )

    print(
        "Question:",
        chunk_test_question
    )

    retrieved_chunks = retrieve_chunks(
        question=chunk_test_question,
        collection_name=experiment["collection_name"],
        top_k=3
    )

    show_retrieved_chunks(
        retrieved_chunks
    )

    prompt = create_prompt(
        question=chunk_test_question,
        retrieved_chunks=retrieved_chunks
    )

    final_answer = generate_answer(
        prompt
    )

    print("\nFinal Answer:")
    print(final_answer)

    best_score = retrieved_chunks[0]["score"]

    chunk_experiment_results.append(
        {
            "experiment": experiment["experiment"],
            "chunk_size": experiment["chunk_size"],
            "overlap": experiment["chunk_overlap"],
            "best_score": round(best_score, 4),
            "answer": final_answer
        }
    )


Experiment: A
Chunk Size: 300
Chunk Overlap: 50
Question: What is supervised learning and how should a model be evaluated on unseen data?

Retrieved Chunks

Rank: 1
Similarity Score: 0.5549
File Name: artificial_intelligence.txt
Chunk Number: 7
Text:
the relationship between the image features and the labels. After training, the model is tested using new data that it has not seen before.

Unsupervised learning uses data without known labels. The model tries to discover patterns, similarities, or groups inside the data. For example, a business
----------------------------------------------------------------------

Rank: 2
Similarity Score: 0.5442
File Name: machine_learning.txt
Chunk Number: 8
Text:
validation set helps developers compare settings and select a suitable model. The test set measures the final model's performance on unseen data. Unseen data is important because a real system must work correctly on new examples, not only on the examples used during training.
--------------

In [7]:
import pandas as pd


chunk_comparison_table = pd.DataFrame(
    chunk_experiment_results
)


chunk_comparison_table = chunk_comparison_table.rename(
    columns={
        "experiment": "Experiment",
        "chunk_size": "Chunk Size",
        "overlap": "Chunk Overlap",
        "best_score": "Best Similarity Score",
        "answer": "Final Answer"
    }
)


chunk_comparison_table

,Experiment,Chunk Size,Chunk Overlap,Best Similarity Score,Final Answer
0,A,300,50,0.5549,Supervised learning is a type of machine learn...
1,B,500,50,0.5535,Supervised learning is a type of machine learn...
2,C,800,100,0.5444,Supervised learning is a type of machine learn...


### Conclusion of the Chunk-Size Experiment

Experiment A produced the highest individual similarity score, but the smaller chunks sometimes separated related information into different sections.

Experiment C provided more surrounding context, but the larger chunks also included additional information that was not directly necessary for the question.

Experiment B, using a chunk size of 500 characters and an overlap of 50 characters, provided the best overall balance. Its retrieved chunks were focused, complete, and sufficient for generating a clear answer. Therefore, the 500/50 configuration was selected for the remaining experiments.

## Task 3 — Compare Different Top-K Values

This experiment compares Top-1, Top-3, and Top-5 retrieval using the same question and the selected 500/50 chunk configuration.

In [8]:
BEST_COLLECTION = "rag_documents_chunk_500"


top_k_question = (
    "Why is data cleaning important in a data science project?"
)


# Store the experiment results
top_k_results = []


for top_k_value in [1, 3, 5]:

    print("\n" + "=" * 80)

    print("Top-K Value:", top_k_value)

    print("Question:")
    print(top_k_question)

    retrieved_chunks = retrieve_chunks(
        question=top_k_question,
        collection_name=BEST_COLLECTION,
        top_k=top_k_value
    )

    show_retrieved_chunks(
        retrieved_chunks
    )

    prompt = create_prompt(
        question=top_k_question,
        retrieved_chunks=retrieved_chunks
    )

    final_answer = generate_answer(
        prompt
    )

    print("\nFinal Answer:")
    print(final_answer)

    similarity_scores = []

    for chunk in retrieved_chunks:
        similarity_scores.append(
            chunk["score"]
        )

    average_score = (
        sum(similarity_scores) /
        len(similarity_scores)
    )

    top_k_results.append(
        {
            "top_k": top_k_value,
            "retrieved_chunks": len(retrieved_chunks),
            "best_score": round(
                similarity_scores[0],
                4
            ),
            "average_score": round(
                average_score,
                4
            ),
            "answer": final_answer
        }
    )


Top-K Value: 1
Question:
Why is data cleaning important in a data science project?

Retrieved Chunks

Rank: 1
Similarity Score: 0.6999
File Name: data_science.txt
Chunk Number: 2
Text:
such as text, images, audio, and videos.

Data cleaning is an important part of data science. Real datasets may contain missing values, duplicate records, incorrect information, and inconsistent formats. If these problems are not corrected, the analysis may produce misleading results. Data cleaning may include removing duplicates, correcting errors, handling missing values, and converting data into a consistent format.
----------------------------------------------------------------------

Final Answer:
Data cleaning is important in a data science project because it addresses issues such as missing values, duplicate records, incorrect information, and inconsistent formats. If these problems are not corrected, the analysis may produce misleading results.

Top-K Value: 3
Question:
Why is data cleaning imp

In [9]:
top_k_comparison_table = pd.DataFrame(
    top_k_results
)


top_k_comparison_table = top_k_comparison_table.rename(
    columns={
        "top_k": "Top-K",
        "retrieved_chunks": "Retrieved Chunks",
        "best_score": "Best Similarity Score",
        "average_score": "Average Similarity Score",
        "answer": "Final Answer"
    }
)


top_k_comparison_table

,Top-K,Retrieved Chunks,Best Similarity Score,Average Similarity Score,Final Answer
0,1,1,0.6999,0.6999,Data cleaning is important in a data science p...
1,3,3,0.6999,0.6280,Data cleaning is important in a data science p...
2,5,5,0.6999,0.5896,Data cleaning is important in a data science p...


### Conclusion of the Top-K Experiment

Top-1 retrieved the most relevant chunk and generated a correct and concise answer. However, using only one chunk may miss supporting information for more complex questions.

Top-3 added useful supporting context while keeping the retrieved information focused. It produced a clear and reliable answer.

Top-5 provided more context, but some lower-ranked chunks were less directly related to the question. Adding more chunks also increased the amount of context sent to the language model.

Therefore, Top-3 was selected for the remaining experiments because it provides a good balance between sufficient context and retrieval noise.

## Task 4 — Evaluate Retrieval Quality

The RAG system is evaluated using ten questions with different difficulty levels.

In [10]:
evaluation_questions = [
    {
        "question": "What is Python mainly used for?",
        "difficulty": "Easy",
        "expected_file": "python.txt"
    },
    {
        "question": "What is supervised learning?",
        "difficulty": "Easy",
        "expected_file": "machine_learning.txt"
    },
    {
        "question": "Why is data cleaning important?",
        "difficulty": "Easy",
        "expected_file": "data_science.txt"
    },
    {
        "question": "Why do football teams use formations?",
        "difficulty": "Medium",
        "expected_file": "football.txt"
    },
    {
        "question": "How can travel improve cultural understanding?",
        "difficulty": "Medium",
        "expected_file": "travel.txt"
    },
    {
        "question": "What role do rockets play in space exploration?",
        "difficulty": "Medium",
        "expected_file": "space.txt"
    },
    {
        "question": "Why are primary sources important in history?",
        "difficulty": "Medium",
        "expected_file": "history.txt"
    },
    {
        "question": (
            "How do vaccination and sanitation help "
            "protect public health?"
        ),
        "difficulty": "Difficult",
        "expected_file": "health.txt"
    },
    {
        "question": (
            "Why are testing, version control, and modularity "
            "important in programming?"
        ),
        "difficulty": "Difficult",
        "expected_file": "programming.txt"
    },
    {
        "question": (
            "How are natural language processing and "
            "computer vision different?"
        ),
        "difficulty": "Difficult",
        "expected_file": "artificial_intelligence.txt"
    }
]


print(
    "Total evaluation questions:",
    len(evaluation_questions)
)

Total evaluation questions: 10


In [11]:
evaluation_results = []


for question_number, item in enumerate(
    evaluation_questions,
    start=1
):

    print("\n" + "=" * 90)

    print("Question Number:", question_number)
    print("Difficulty:", item["difficulty"])
    print("Question:")
    print(item["question"])

    retrieved_chunks = retrieve_chunks(
        question=item["question"],
        collection_name=BEST_COLLECTION,
        top_k=3
    )

    show_retrieved_chunks(
        retrieved_chunks
    )

    prompt = create_prompt(
        question=item["question"],
        retrieved_chunks=retrieved_chunks
    )

    final_answer = generate_answer(
        prompt
    )

    print("\nFinal Answer:")
    print(final_answer)

    retrieved_files = []

    for chunk in retrieved_chunks:
        retrieved_files.append(
            chunk["file_name"]
        )

    if item["expected_file"] in retrieved_files:
        retrieval_correct = "Yes"
    else:
        retrieval_correct = "No"

    best_score = retrieved_chunks[0]["score"]

    evaluation_results.append(
        {
            "Question": item["question"],
            "Difficulty": item["difficulty"],
            "Expected File": item["expected_file"],
            "Retrieved Files": ", ".join(retrieved_files),
            "Best Similarity Score": round(best_score, 4),
            "Retrieval Correct?": retrieval_correct,
            "Final Answer": final_answer
        }
    )


Question Number: 1
Difficulty: Easy
Question:
What is Python mainly used for?

Retrieved Chunks

Rank: 1
Similarity Score: 0.618
File Name: python.txt
Chunk Number: 2
Text:
community that creates tutorials, documentation, libraries, and development tools.

Python is used in many areas of technology. It is widely used for web development, artificial intelligence, machine learning, data science, automation, scientific computing, and software development. Developers can also use Python to create desktop applications, connect with databases, work with web APIs, and process different types of files. Python is especially useful for automating repetitive tasks. For
----------------------------------------------------------------------

Rank: 2
Similarity Score: 0.6103
File Name: python.txt
Chunk Number: 0
Text:
Python is a high-level and general-purpose programming language. It is well known for its simple and readable syntax. Python was created by Guido van Rossum and was first released in 

In [12]:
manual_retrieval_judgment = [
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "No"
]


manual_answer_judgment = [
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "No"
]


prompt_followed_judgment = [
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes",
    "Yes"
]


for index in range(len(evaluation_results)):

    evaluation_results[index][
        "Retrieved Chunks Relevant?"
    ] = manual_retrieval_judgment[index]

    evaluation_results[index][
        "Final Answer Correct?"
    ] = manual_answer_judgment[index]

    evaluation_results[index][
        "Prompt Followed?"
    ] = prompt_followed_judgment[index]


evaluation_table = pd.DataFrame(
    evaluation_results
)


evaluation_table = evaluation_table[
    [
        "Question",
        "Difficulty",
        "Best Similarity Score",
        "Retrieved Chunks Relevant?",
        "Final Answer Correct?",
        "Prompt Followed?"
    ]
]


evaluation_table

,Question,Difficulty,Best Similarity Score,Retrieved Chunks Relevant?,Final Answer Correct?,Prompt Followed?
0,What is Python mainly used for?,Easy,0.6180,Yes,Yes,Yes
1,What is supervised learning?,Easy,0.6314,Yes,Yes,Yes
2,Why is data cleaning important?,Easy,0.6578,Yes,Yes,Yes
3,Why do football teams use formations?,Medium,0.6861,Yes,Yes,Yes
4,How can travel improve cultural understanding?,Medium,0.6447,Yes,Yes,Yes
5,What role do rockets play in space exploration?,Medium,0.6125,Yes,Yes,Yes
6,Why are primary sources important in history?,Medium,0.5745,Yes,Yes,Yes
7,How do vaccination and sanitation help protect...,Difficult,0.5456,Yes,Yes,Yes
8,"Why are testing, version control, and modulari...",Difficult,0.5764,Yes,Yes,Yes
9,How are natural language processing and comput...,Difficult,0.3580,No,No,Yes


In [13]:
correct_retrievals = (
    evaluation_table[
        "Retrieved Chunks Relevant?"
    ] == "Yes"
).sum()


retrieval_accuracy = (
    correct_retrievals /
    len(evaluation_table)
) * 100


correct_answers = (
    evaluation_table[
        "Final Answer Correct?"
    ] == "Yes"
).sum()


answer_accuracy = (
    correct_answers /
    len(evaluation_table)
) * 100


print(
    "Retrieval Accuracy:",
    f"{retrieval_accuracy:.1f}%"
)

print(
    "Final Answer Accuracy:",
    f"{answer_accuracy:.1f}%"
)

Retrieval Accuracy: 90.0%
Final Answer Accuracy: 90.0%


### Evaluation Conclusion

The RAG system successfully retrieved relevant chunks and generated correct answers for 9 out of 10 questions. Therefore, both retrieval accuracy and final-answer accuracy were 90%.

The final difficult question about natural language processing and computer vision was not answered correctly. Although the retrieved chunks came from the expected artificial intelligence document, they discussed neural networks, deep learning, and general AI instead of directly explaining the difference between NLP and computer vision.

The relatively low similarity scores also indicated that the retrieved chunks were weak matches. The strict prompt prevented the model from inventing an unsupported answer and correctly produced the insufficient-information message.

## Task 5 — Handle Unknown Questions

In [14]:
unknown_questions = [
    "What is Docker?",
    "What is Kubernetes?",
    "Who won the FIFA World Cup in 2022?"
]


unknown_question_results = []


for question_number, question in enumerate(
    unknown_questions,
    start=1
):

    print("\n" + "=" * 90)

    print(
        "Unknown Question Number:",
        question_number
    )

    print("Question:")
    print(question)

    retrieved_chunks = retrieve_chunks(
        question=question,
        collection_name=BEST_COLLECTION,
        top_k=3
    )

    show_retrieved_chunks(
        retrieved_chunks
    )

    prompt = create_prompt(
        question=question,
        retrieved_chunks=retrieved_chunks
    )

    final_answer = generate_answer(
        prompt
    )

    print("\nFinal Answer:")
    print(final_answer)

    expected_unknown_answer = (
        "I don't have enough information to answer this question "
        "based on the available documents."
    )

    if final_answer == expected_unknown_answer:
        hallucination_prevented = "Yes"
    else:
        hallucination_prevented = "No"

    unknown_question_results.append(
        {
            "Question": question,
            "Best Similarity Score": round(
                retrieved_chunks[0]["score"],
                4
            ),
            "Hallucination Prevented?": hallucination_prevented,
            "Final Answer": final_answer
        }
    )


Unknown Question Number: 1
Question:
What is Docker?

Retrieved Chunks

Rank: 1
Similarity Score: 0.2725
File Name: programming.txt
Chunk Number: 4
Text:
make software easier to read, test, and maintain.

Debugging is the process of finding and correcting errors in software. A developer may use error messages, print statements, logs, and debugging tools to understand a problem. Testing checks whether the program behaves as expected. Unit tests examine small functions, while integration tests check whether different parts work together.
----------------------------------------------------------------------

Rank: 2
Similarity Score: 0.2554
File Name: programming.txt
Chunk Number: 8
Text:
folders from being uploaded to GitHub.

Good software development requires clear requirements, readable code, testing, documentation, error handling, security, and regular maintenance. Programming is not only typing code. It is a systematic process of understanding a problem, designing a solution, chec

In [15]:
unknown_question_table = pd.DataFrame(
    unknown_question_results
)

unknown_question_table

,Question,Best Similarity Score,Hallucination Prevented?,Final Answer
0,What is Docker?,0.2725,Yes,I don't have enough information to answer this...
1,What is Kubernetes?,0.2169,Yes,I don't have enough information to answer this...
2,Who won the FIFA World Cup in 2022?,0.2888,Yes,I don't have enough information to answer this...


### Explanation of Unknown-Question Handling

Qdrant always tries to return the closest available chunks, even when the exact answer is not present in the documents. Therefore, retrieving a chunk does not automatically mean that the chunk contains the answer.

For example, a Docker or Kubernetes question may retrieve general programming chunks, while a World Cup question may retrieve general football chunks. These results may be related to the topic, but they do not contain the required answer.

The strict prompt prevents the language model from using outside knowledge or guessing. When the answer is not supported by the retrieved context, the model returns:

> I don't have enough information to answer this question based on the available documents.

Hallucination can be reduced by using a strict prompt, checking similarity scores, reviewing retrieved chunks, applying a tested score threshold, and improving the dataset and retrieval method.

## Task 6 — Improve the Prompt

This experiment compares a weak prompt with the improved strict prompt.

The test includes:

- One valid question whose answer exists in the documents
- One invalid question whose answer does not exist in the documents

The purpose is to observe whether clear prompt rules reduce unsupported or hallucinated answers.

In [16]:
import importlib
import src.prompt as prompt_module


importlib.reload(prompt_module)


prompt_test_questions = [
    {
        "type": "Valid Question",
        "question": "Why is Python considered easy to read?"
    },
    {
        "type": "Invalid Question",
        "question": "What is Docker used for?"
    }
]


prompt_comparison_results = []


for item in prompt_test_questions:

    print("\n" + "=" * 90)

    print("Question Type:", item["type"])
    print("Question:")
    print(item["question"])

    retrieved_chunks = retrieve_chunks(
        question=item["question"],
        collection_name=BEST_COLLECTION,
        top_k=3
    )

    show_retrieved_chunks(
        retrieved_chunks
    )

    weak_prompt = prompt_module.create_weak_prompt(
        question=item["question"],
        retrieved_chunks=retrieved_chunks
    )

    weak_answer = generate_answer(
        weak_prompt
    )

    strict_prompt = prompt_module.create_prompt(
        question=item["question"],
        retrieved_chunks=retrieved_chunks
    )

    strict_answer = generate_answer(
        strict_prompt
    )

    print("\nWeak Prompt Answer:")
    print(weak_answer)

    print("\nStrict Prompt Answer:")
    print(strict_answer)

    # Save the results
    prompt_comparison_results.append(
        {
            "Question Type": item["type"],
            "Question": item["question"],
            "Weak Prompt Answer": weak_answer,
            "Strict Prompt Answer": strict_answer
        }
    )


Question Type: Valid Question
Question:
Why is Python considered easy to read?

Retrieved Chunks

Rank: 1
Similarity Score: 0.6212
File Name: python.txt
Chunk Number: 0
Text:
Python is a high-level and general-purpose programming language. It is well known for its simple and readable syntax. Python was created by Guido van Rossum and was first released in 1991. It is suitable for both beginners and experienced developers. The language uses clear English-like statements, which makes Python code easier to read and understand. Python also uses indentation to separate blocks of code. This helps programmers keep their code clean and organized.
----------------------------------------------------------------------

Rank: 2
Similarity Score: 0.5418
File Name: python.txt
Chunk Number: 1
Text:
programmers keep their code clean and organized.

Python is an interpreted programming language. A Python interpreter reads and executes the instructions in a program. Developers can run and test Python 

In [17]:
prompt_comparison_table = pd.DataFrame(
    prompt_comparison_results
)

prompt_comparison_table

,Question Type,Question,Weak Prompt Answer,Strict Prompt Answer
0,Valid Question,Why is Python considered easy to read?,Python is considered easy to read due to its s...,Python is considered easy to read because it u...
1,Invalid Question,What is Docker used for?,The context provided does not mention Docker s...,I don't have enough information to answer this...


### Conclusion of the Prompt Comparison

For the valid question, both the weak prompt and the strict prompt generated a correct answer because the required information was available in the retrieved context.

For the invalid Docker question, the weak prompt did not invent an answer, but it responded in its own wording. The strict prompt followed the required instruction and returned the exact insufficient-information message.

The strict prompt is more reliable because it clearly tells the language model not to use outside knowledge, not to guess, and to use a fixed response when the answer is unavailable. Therefore, the improved strict prompt was selected for the final RAG system.

## Task 7 — Reflection Questions

### 1. Why do we split documents into chunks?

We split large documents into smaller chunks so that the retrieval system can find the most relevant section of a document. Sending only relevant chunks also reduces unnecessary context and makes the final answer more focused.

### 2. Why do we generate embeddings?

Embeddings convert text into numerical vectors that represent semantic meaning. These vectors allow the system to compare a user question with document chunks and find text with similar meaning.

### 3. Why do we use a vector database?

A vector database stores embeddings together with their text and metadata. It can quickly compare a question vector with many stored vectors and return the most similar document chunks.

### 4. What happens if the chunk size is too small?

If the chunk size is too small, related information may be divided into separate chunks. A retrieved chunk may contain only part of the answer and may not provide enough context for the language model.

### 5. What happens if the chunk size is too large?

If the chunk size is too large, one chunk may contain both relevant and irrelevant information. Large chunks also increase the amount of text sent to the language model and may make retrieval less focused.

### 6. Why do we retrieve multiple chunks instead of only one?

One chunk may not contain all the information required to answer a question. Retrieving multiple chunks can provide definitions, explanations, and supporting details. However, retrieving too many chunks may introduce irrelevant information.

### 7. What happens when irrelevant chunks are retrieved?

Irrelevant chunks can confuse the language model and may produce an incomplete, incorrect, or unsupported answer. Therefore, retrieved chunks and similarity scores should be reviewed carefully.

### 8. How can retrieval quality be improved?

Retrieval quality can be improved by using better documents, suitable chunk sizes and overlaps, a strong embedding model, appropriate Top-K values, metadata filtering, query rewriting, score thresholds, and regular evaluation of retrieved results.

In [19]:
from src.vector_store import close_qdrant_client


close_qdrant_client()

Qdrant client closed successfully.
